# Vocabulary growth — novelty and richness

How rich and how novel is the language over time? We look at the mix of
n-gram lengths, how many *distinct* phrases appear each day, and when brand
new phrases first enter the stream.

*SQL is kept inline in this notebook.*

In [ ]:
import os
from pathlib import Path

import pandas as pd
import plotly.express as px
import psycopg
from dotenv import load_dotenv

# Connection settings come from the repository root .env
load_dotenv(Path("..") / ".env")

conninfo = psycopg.conninfo.make_conninfo(
    host=os.environ["POSTGRES_HOST"],
    port=int(os.getenv("POSTGRES_PORT", "5432")),
    user=os.environ["POSTGRES_USER"],
    password=os.environ["POSTGRES_PASSWORD"],
    dbname=os.environ["POSTGRES_DATABASE"],
    sslmode=os.getenv("PGSSLMODE", "require"),
)


def q(sql: str, **params) -> pd.DataFrame:
    """Run a query and return a DataFrame. Params are passed safely to psycopg."""
    with psycopg.connect(conninfo) as conn, conn.cursor() as cur:
        cur.execute(sql, params or None)
        cols = [d.name for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)


print("Ready — connected settings for", os.environ["POSTGRES_HOST"])


## n-gram length distribution
How many single words vs. bigrams vs. longer phrases.

In [ ]:
lengths = q("""
    SELECT n_gram, COUNT(*) AS n
    FROM ngrams
    GROUP BY n_gram
    ORDER BY n_gram
""")

fig = px.bar(lengths, x="n_gram", y="n", text_auto=True,
             title="Distribution of n-gram lengths")
fig.update_layout(height=380, xaxis_title="n-gram length",
                  yaxis_title="occurrences")
fig.show()

## Vocabulary richness per day
Distinct phrases vs. total occurrences each day. When the two lines diverge, the stream is repeating itself; when they track, it is exploring.

In [ ]:
richness = q("""
    SELECT date_trunc('day', timestamp) AS day,
           COUNT(*)               AS occurrences,
           COUNT(DISTINCT words)  AS distinct_phrases
    FROM ngrams
    GROUP BY 1
    ORDER BY 1
""")

long = richness.melt(
    id_vars="day", value_vars=["occurrences", "distinct_phrases"],
    var_name="metric", value_name="count",
)
fig = px.line(long, x="day", y="count", color="metric",
              title="Daily volume vs. distinct vocabulary")
fig.update_layout(height=440, xaxis_title="", yaxis_title="")
fig.show()

## Emergence of new phrases
For every phrase, take the day it was *first* seen, then count how many
phrases debuted each day — a proxy for how much genuinely new language the
stream introduces over time.

In [ ]:
newborn = q("""
    WITH first_seen AS (
        SELECT words, MIN(timestamp) AS first_ts
        FROM ngrams
        GROUP BY words
    )
    SELECT date_trunc('day', first_ts) AS day,
           COUNT(*)                    AS new_phrases
    FROM first_seen
    GROUP BY 1
    ORDER BY 1
""")

fig = px.bar(newborn, x="day", y="new_phrases",
             title="New phrases entering the stream per day")
fig.update_layout(height=420, xaxis_title="", yaxis_title="first-seen phrases")
fig.show()